# Earthquake catalog quality-control — synthetic example

This notebook creates **artificial events** with a fixed random seed. It illustrates structural validation, duplicate detection, event plots and summary counts, NOT a real seismicity study or PSHA.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(2026)
n=45
dates = pd.date_range("2025-01-01",periods=n,freq="10D")
df = pd.DataFrame({"event_id": [f"SYN-{i:03d}" for i in range(n)],"date": dates,"latitude": -5 + rng.normal(0, 1., n),"longitude": -74 + rng.normal(0, 1.5, n),"depth_km": rng.uniform(2, 75, n),"magnitude_Mw": rng.uniform(2.5, 5.8, n)})
df.loc[4,"depth_km"]=np.nan
df = pd.concat([df,df.iloc[[1]]],ignore_index=True) # synthetic duplicate event

In [ ]:
duplicates = df.duplicated("event_id",keep=False)
qc = pd.DataFrame({"check": ["Total rows","Unique IDs","Rows with duplicated event ID","Rows with any missing values"],"count":[len(df),df.event_id.nunique(),int(duplicates.sum()),int(df.isna().any(axis=1).sum())]})
print(qc.to_string(index=False))
clean = df.drop_duplicates(subset=["event_id"]).dropna().copy()
print(f"Valid rows after basic checks: {len(clean)}")

In [ ]:
fig,ax=plt.subplots(figsize=(7,4.8))
ax.scatter(clean["magnitude_Mw"],clean["depth_km"],alpha=.75)
ax.set_xlabel("Synthetic magnitude Mw");ax.set_ylabel("Synthetic hypocentral depth [km]");ax.grid(alpha=.25)
plt.tight_layout();plt.show()

## Remaining work for real catalogs

Magnitude-scale harmonization, spatial/temporal completeness, network changes, declustering, duplicated-event provenance and tectonic model selection all require additional documented decisions. **No seismic hazard calculation is performed.**